# TypedDict를 사용 하는 이유
정적 타입 검사기

- 타입 안정성: 잠재적인 버그 방지(타입 검사)
- 코드 가독성: 딕셔너리의 구조를 명확하게 정의
- IDE 지원: 자동완성 및 타입 힌트 제공
- 문서화: 코드자체가 문서의 역할

# Annotated를 사용하는 이유
추가 정보 제공(타입 힌트) / 문서화

- 추가 정보 제공 타입 힌트에 메타데이터를 추가하여 더 상세한 정보를 제공.
- 문서화 코드 자체에 추가 설명을 포함시켜 문서화 효과.
- 유효성 검사 특정 라이브러리(예: Pydantic)와 함께 사용하여 데이터 유효성 검사
- 프레임워크 지원 일부 프레임워크(예: LangGraph)에서는 `Annotated`를 사용하여 특별한 동작을 정의.

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [2]:
# Dict와 TypedDict의 차이점 예시

from re import I
from typing import Dict, TypedDict

simple_dict: Dict[str, str] = {
    "name": "Jaeho",
    "age": "31",
    "job": "Developer",
}

class Person(TypedDict):
    name: str
    age: int
    job: str


typed_dict: Person = {"name": "재호", "age": 31, "job": "개발자"}

In [ ]:
print(simple_dict)
print(typed_dict)

{'name': 'Jaeho', 'age': '31', 'job': 'Developer'}
{'name': '재호', 'age': 31, 'job': '개발자'}


In [13]:
# dict의 경우
simple_dict["age"] = 35  # 문자열에서 정수로 변경되어도 오류 없음
simple_dict["new_field"] = "추가 정보"  # 새로운 필드 추가 가능

# TypedDict의 경우
typed_dict["age"] = 35  # 정수형으로 올바르게 사용
typed_dict["age"] = "35"  # 타입 체커가 오류를 감지함
typed_dict["new_field"] = (
    "추가 정보"  # 타입 체커가 정의되지 않은 키라고 오류를 발생시킴
)

In [16]:
print(simple_dict)
print(typed_dict)

{'name': 'Jaeho', 'age': 35, 'job': 'Developer', 'new_field': '추가 정보'}
{'name': '재호', 'age': '35', 'job': '개발자', 'new_field': '추가 정보'}


In [17]:
from typing import Annotated

name: Annotated[str, "사용자 이름"]
age: Annotated[int, "사용자 나이 (0-150)"]

In [23]:
from typing import Annotated, List
from pydantic import Field, BaseModel, ValidationError


class Employee(BaseModel):
    id: Annotated[int, Field(..., description="직원 ID")]
    name: Annotated[str, Field(..., min_length=3, max_length=50, description="이름")]
    age: Annotated[int, Field(gt=18, lt=65, description="나이 (19-64세)")]
    salary: Annotated[
        float, Field(gt=0, lt=10000, description="연봉 (단위: 만원, 최대 10억)")
    ]
    skills: Annotated[
        List[str], Field(min_items=1, max_items=10, description="보유 기술 (1-10개)")
    ]


# 유효한 데이터로 인스턴스 생성
try:
    valid_employee = Employee(
        id=1, name="김재호", age=30, salary=9000, skills=["Python", "LangChain"]
    )
    print("유효한 직원 데이터:", valid_employee)
except ValidationError as e:
    print("유효성 검사 오류:", e)



유효한 직원 데이터: id=1 name='김재호' age=30 salary=9000.0 skills=['Python', 'LangChain']


In [24]:
# 유효하지 않은 데이터로 인스턴스 생성 시도
try:
    invalid_employee = Employee(
        name="테디",  # 이름이 너무 짧음
        age=17,  # 나이가 범위를 벗어남
        salary=20000,  # 급여가 범위를 벗어남
        skills="Python",  # 리스트가 아님
    )
except ValidationError as e:
    print("유효성 검사 오류:")
    for error in e.errors():
        print(f"- {error['loc'][0]}: {error['msg']}")

유효성 검사 오류:
- id: Field required
- name: String should have at least 3 characters
- age: Input should be greater than 18
- salary: Input should be less than 10000
- skills: Input should be a valid list


### LangGraph에서의 사용(add_messages)

`add_messages` 는 LangGraph 에서 메시지를 리스트에 추가하는 함수입니다.

## add_messages

`messages` 키는 [`add_messages`](https://langchain-ai.github.io/langgraph/reference/graphs/?h=add+messages#add_messages) 리듀서 함수

- 주석이 없는 상태 키는 각 업데이트에 의해 덮어쓰여져 가장 최근의 값이 저장.
   - 두 개의 메시지 리스트를 병합.
   - 기본적으로 "append-only" 상태를 유지.
   - 동일한 ID를 가진 메시지가 있을 경우, 새 메시지로 기존 메시지를 대체.

   - `left` (Messages): 기본 메시지 리스트
   - `right` (Messages): 병합할 메시지 리스트 또는 단일 메시지
   - `Messages`: `right`의 메시지들이 `left`에 병합된 새로운 메시지 리스트   


In [25]:
from typing import Annotated, TypedDict
from langgraph.graph import add_messages

class MyData(TypedDict):
    messages: Annotated[list, add_messages]

In [26]:
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.graph import add_messages

# 기본 사용 예시
msgs1 = [HumanMessage(content="안녕하세요?", id="1")]
msgs2 = [AIMessage(content="반갑습니다~", id="2")]

result1 = add_messages(msgs1, msgs2)
print(result1)

[HumanMessage(content='안녕하세요?', additional_kwargs={}, response_metadata={}, id='1'), AIMessage(content='반갑습니다~', additional_kwargs={}, response_metadata={}, id='2')]


동일한 id가 있을때 대체.

In [31]:
# 동일한 ID를 가진 메시지 대체 예시
msgs1 = [HumanMessage(content="안녕하세요?", id="1")]
msgs2 = [HumanMessage(content="반갑습니다~", id="1")]

result2 = add_messages(msgs1, msgs2)
print(result2)

[HumanMessage(content='반갑습니다~', additional_kwargs={}, response_metadata={}, id='1')]


In [33]:
# 동일한 ID를 가진 메시지 대체 예시
msgs1 = [HumanMessage(content="안녕하세요?", id="1")]
msgs2 = [HumanMessage(content="반갑습니다~", id="2")]

result2 = add_messages(msgs1, msgs2)
print(result2)

[HumanMessage(content='안녕하세요?', additional_kwargs={}, response_metadata={}, id='1'), HumanMessage(content='반갑습니다~', additional_kwargs={}, response_metadata={}, id='2')]


In [34]:
msgs3 = [AIMessage(content="반갑습니다~", id="1")]

result3 = add_messages(result2, msgs3)
print(result3)

[AIMessage(content='반갑습니다~', additional_kwargs={}, response_metadata={}, id='1'), HumanMessage(content='반갑습니다~', additional_kwargs={}, response_metadata={}, id='2')]


In [35]:
msgs4 = [AIMessage(content="안녕하세요?", id="2")]

result4 = add_messages(result3, msgs4)
print(result3)

[AIMessage(content='반갑습니다~', additional_kwargs={}, response_metadata={}, id='1'), HumanMessage(content='반갑습니다~', additional_kwargs={}, response_metadata={}, id='2')]
